**Flight Price Prediction Using Machine Learning**

Importing Libraries

In [ ]:
from IPython import get_ipython
from IPython.display import display
# %% [markdown]
# Importing Libraries
# %%

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import r2_score
from math import sqrt
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from prettytable import PrettyTable

from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

from sklearn.metrics import mean_absolute_percentage_error # Import the function


Reading the Training Data of our Dataset

In [ ]:
import pandas as pd

# Read the Excel file directly
train_df = pd.read_excel("/content/Data_Train.xlsx")
print(train_df.head(10))

Exploratory Data Analysis (EDA)

In [ ]:
train_df.columns

Here we can get more information about our dataset

In [ ]:
train_df.info()


To know more about the dataset

In [ ]:
train_df.describe()


Now while using the IsNull function we will gonna see the number of null values in our dataset

In [ ]:
train_df.isnull().head()

Now while using the IsNull function and sum function we will gonna see the number of null values in our dataset

In [ ]:
train_df.isnull().sum()

Dropping NAN values

In [ ]:
train_df.dropna(inplace = True)

Duplicate values

In [ ]:
train_df[train_df.duplicated()].head()

Here we will be removing those repeated values from the dataset and keeping the in-place attribute to be true so that there will be no changes

In [ ]:
train_df.drop_duplicates(keep='first',inplace=True)
train_df.head()

In [ ]:
train_df.shape

Checking the Additional_info column and having the count of unique types of values.

In [ ]:
train_df["Additional_Info"].value_counts()

Checking the different Airlines

In [ ]:
train_df["Airline"].unique()

Checking the different Airline Routes

In [ ]:
train_df["Route"].unique()

Now let’s look at our testing dataset

In [ ]:
import pandas as pd # Importing the pandas library with the alias 'pd'

test_df = pd.read_excel("/content/test.xlsx")
test_df.head(10)

Now here we will be looking at the kind of columns our testing data has.

In [ ]:
test_df.columns

Information about the dataset

In [ ]:
test_df.info()

To know more about the testing dataset

In [ ]:
test_df.describe()

Now while using the IsNull function and sum function we will gonna see the number of null values in our testing data

In [ ]:
test_df.isnull().sum()

Data Visualization of Flight Price Prediction using Machine Learning

Plotting Price vs Airline plot

In [ ]:
import seaborn as sns # Importing the seaborn library
import matplotlib.pyplot as plt # Make sure to import matplotlib.pyplot

sns.catplot(y = "Price", x = "Airline", data = train_df.sort_values("Price", ascending = False), kind="boxen", height = 8, aspect = 3)
plt.show()

Plotting Violin plot for Price vs Source

In [ ]:
sns.catplot(y = "Price", x = "Source", data = train_df.sort_values("Price", ascending = False), kind="violin", height = 4, aspect = 3)
plt.show()

Plotting Box plot for Price vs Destination

In [ ]:
sns.catplot(y = "Price", x = "Destination", data = train_df.sort_values("Price", ascending = False), kind="box", height = 4, aspect = 3)
plt.show()

Feature Engineering

Let’s see our processed data first

In [ ]:
train_df.head()

Here first we are dividing the features and labels and then converting the hours in minutes.

In [ ]:
train_df['Duration'] = train_df['Duration'].str.replace("h", '*60').str.replace(' ','+').str.replace('m','*1').apply(eval)
test_df['Duration'] = test_df['Duration'].str.replace("h", '*60').str.replace(' ','+').str.replace('m','*1').apply(eval)

Here we are organizing the format of the date of journey in our dataset for better preprocessing in the model stage.

In [ ]:
train_df["Journey_day"] = train_df['Date_of_Journey'].str.split('/').str[0].astype(int)
train_df["Journey_month"] = train_df['Date_of_Journey'].str.split('/').str[1].astype(int)
train_df.drop(["Date_of_Journey"], axis = 1, inplace = True)

Here we are converting departure time into hours and minutes

In [ ]:
train_df["Dep_hour"] = pd.to_datetime(train_df["Dep_Time"]).dt.hour
train_df["Dep_min"] = pd.to_datetime(train_df["Dep_Time"]).dt.minute
train_df.drop(["Dep_Time"], axis = 1, inplace = True)

Similarly we are converting the arrival time into hours and minutes.

In [ ]:
train_df["Arrival_hour"] = pd.to_datetime(train_df.Arrival_Time).dt.hour
train_df["Arrival_min"] = pd.to_datetime(train_df.Arrival_Time).dt.minute
train_df.drop(["Arrival_Time"], axis = 1, inplace = True)

Now after final preprocessing let’s see our dataset

In [ ]:
train_df.head()

Plotting Bar chart for Months (Duration) vs Number of Flights

In [ ]:
plt.figure(figsize = (10, 5))
plt.title('Count of flights month wise')
ax=sns.countplot(x = 'Journey_month', data = train_df)
plt.xlabel('Month')
plt.ylabel('Count of flights')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+0.25, p.get_height()+1), va='bottom')

Plotting Bar chart for Types of Airline vs Number of Flights

In [ ]:
plt.figure(figsize = (20,5))
plt.title('Count of flights with different Airlines')
ax=sns.countplot(x = 'Airline', data =train_df)
plt.xlabel('Airline')
plt.ylabel('Count of flights')
plt.xticks(rotation = 45)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+0.25, p.get_height()+1), va='bottom')

Plotting Ticket Prices vs Airlines

In [ ]:
plt.figure(figsize = (15,4))
plt.title('Price VS Airlines')
plt.scatter(train_df['Airline'], train_df['Price'])
plt.xticks
plt.xlabel('Airline')
plt.ylabel('Price of ticket')
plt.xticks(rotation = 90)

Plotting Correlation

In [ ]:
import seaborn as sns # Importing the seaborn library


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming train_df has columns with object (string) dtype
# Select only numerical columns for correlation calculation
numerical_cols = train_df.select_dtypes(include=['number']).columns
numerical_train_df = train_df[numerical_cols]

plt.figure(figsize=(15, 15))
sns.heatmap(numerical_train_df.corr(), annot=True, cmap="RdYlGn")
plt.show() # Add this line to display the heatmap

Dropping the Price column as it is of no use

In [ ]:
data = train_df.drop(["Price"], axis=1)

Dealing with Categorical Data and Numerical Data

In [ ]:
import pandas as pd

# ... (your existing code) ...

# Before the line causing the error, load the data into test_df
test_df = pd.read_excel("/content/test.xlsx")

train_categorical_data = data.select_dtypes(exclude=['int64', 'float','int32'])
train_numerical_data = data.select_dtypes(include=['int64', 'float','int32'])

test_categorical_data = test_df.select_dtypes(exclude=['int64', 'float','int32','int32'])
test_numerical_data  = test_df.select_dtypes(include=['int64', 'float','int32'])
train_categorical_data.head()

Label Encode and Hot Encode for Categorical Columns

In [ ]:
le = LabelEncoder()
train_categorical_data = train_categorical_data.apply(LabelEncoder().fit_transform)

# Convert all columns in test_categorical_data to string type before applying LabelEncoder
test_categorical_data = test_categorical_data.astype(str)
test_categorical_data = test_categorical_data.apply(LabelEncoder().fit_transform)

train_categorical_data.head()

Concatenating both Categorical Data and Numerical Data

In [ ]:
X = pd.concat([train_categorical_data, train_numerical_data], axis=1)
y = train_df['Price']
test_set = pd.concat([test_categorical_data, test_numerical_data], axis=1)
X.head()

In [ ]:
y.head()

Now we will be splitting out our dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)
print("The size of training input is", X_train.shape)
print("The size of training output is", y_train.shape)
print("The size of testing input is", X_test.shape)
print("The size of testing output is", y_test.shape)

Model Building

Ridge Regression

In [ ]:
# Performing GridSearchCV on Ridge Regression
params = {'alpha' : [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000]}
ridge_regressor = GridSearchCV(Ridge(), params, cv = 5, scoring = 'neg_mean_absolute_error', n_jobs = -1)
ridge_regressor.fit(X_train, y_train)

In [ ]:
y_train_pred = ridge_regressor.predict(X_train)
y_test_pred = ridge_regressor.predict(X_test)
print("Train Results for Ridge Regressor Model:")
print("Root Mean Squared Error: ", sqrt(mse(y_train.values, y_train_pred)))
print("Mean Absolute % Error: ", round(mean_absolute_percentage_error(y_train.values, y_train_pred)))
print("R-Squared: ", r2_score(y_train.values, y_train_pred))

In [ ]:
print("Test Results for Ridge Regressor Model:")
print("Root Mean Squared Error: ", sqrt(mse(y_test, y_test_pred)))
print("Mean Absolute % Error: ", round(mean_absolute_percentage_error(y_test, y_test_pred)))
print("R-Squared: ", r2_score(y_test, y_test_pred))

Lasso Regression


In [ ]:
# Performing GridSearchCV on Lasso Regression
params = {'alpha' : [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000]}
lasso_regressor = GridSearchCV(Lasso(), params ,cv = 15,scoring = 'neg_mean_absolute_error', n_jobs = -1)
lasso_regressor.fit(X_train, y_train)

In [ ]:
# Predicting train and test results
y_train_pred = lasso_regressor.predict(X_train)
y_test_pred = lasso_regressor.predict(X_test)
print("Train Results for Lasso Regressor Model:")
print("Root Mean Squared Error: ", sqrt(mse(y_train.values, y_train_pred)))
print("Mean Absolute % Error: ", round(mean_absolute_percentage_error(y_train.values, y_train_pred)))
print("R-Squared: ", r2_score(y_train.values, y_train_pred))

In [ ]:
print("Test Results for Lasso Regressor Model:")
print("Root Mean squared Error: ", sqrt(mse(y_test, y_test_pred)))
print("Mean Absolute % Error: ", round(mean_absolute_percentage_error(y_test, y_test_pred)))
print("R-Squared: ", r2_score(y_test, y_test_pred))

Decision Tree Regression


In [ ]:
# Performing GridSearchCV on Decision Tree Regression
depth = list(range(3,30))
param_grid = dict(max_depth = depth)
tree = GridSearchCV(DecisionTreeRegressor(), param_grid, cv = 10)
tree.fit(X_train,y_train)

In [ ]:
# Predicting train and test results
y_train_pred = tree.predict(X_train)
y_test_pred = tree.predict(X_test)
print("Train Results for Decision Tree Regressor Model:")
print("Root Mean squared Error: ", sqrt(mse(y_train.values, y_train_pred)))
print("Mean Absolute % Error: ", round(mean_absolute_percentage_error(y_train.values, y_train_pred)))
print("R-Squared: ", r2_score(y_train.values, y_train_pred))

In [ ]:
print("Test Results for Decision Tree Regressor Model:")
print("Root Mean Squared Error: ", sqrt(mse(y_test, y_test_pred)))
print("Mean Absolute % Error: ", round(mean_absolute_percentage_error(y_test, y_test_pred)))
print("R-Squared: ", r2_score(y_test, y_test_pred))

In [ ]:
ridge_score = round(ridge_regressor.score(X_train, y_train) * 100, 2)
ridge_score_test = round(ridge_regressor.score(X_test, y_test) * 100, 2)

lasso_score = round(lasso_regressor.score(X_train, y_train) * 100, 2)
lasso_score_test = round(lasso_regressor.score(X_test, y_test) * 100, 2)

decision_score = round(tree.score(X_train, y_train) * 100, 2)
decision_score_test = round(tree.score(X_test, y_test) * 100, 2)

Comparing all the Models


In [ ]:
# Comparing all the models
models = pd.DataFrame({
    'Model': [ 'Ridge Regression', 'Lasso Regression','Decision Tree Regressor'],
    'Score': [ ridge_score, lasso_score, decision_score],
    'Test Score': [ ridge_score_test, lasso_score_test, decision_score_test]})
models.sort_values(by='Test Score', ascending=False)

In [ ]:
# Training = Tr.
# Testing = Te.
x = PrettyTable()
x.field_names = ["Model Name", "Tr. RMSE", "Tr. MA%E", "Tr. R-Squared", "Te. RMSE", "Te. MA%E", "Te. R-Squared",]
x.add_row(['Ridge Regression','3558.67','32','0.42','3457.60','32','0.42'])
x.add_row(['Lasso Regression','3560.85','32','0.41','3459.38','32','0.42'])
x.add_row(['Decision Tree Regressor','853.54','06','0.97','1857.68','10','0.83'])
print(x)